## Import some packages

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import integrate
import os

import ADFWI
from ADFWI.model import AcousticModel
from ADFWI.propagator import AcousticPropagator
from ADFWI.survey import Receiver, SeismicData, Source, Survey
from ADFWI.utils import numpy2tensor, wavelet
from ADFWI.view import plot_damp
project_path = "./data"
for subdir in ("model", "waveform", "survey"):
    os.makedirs(os.path.join(project_path, subdir), exist_ok=True)

## Basic Parameter

In [ ]:
device = "npu:0"         # Specify the CPU/GPU/NPU device
dtype = torch.float32     # Set data type to 32-bit floating point
backend = ADFWI.set_backend(device, dtype=dtype)
ox, oz = 0, 0             # Origin coordinates for x and z directions
nz, nx = 120, 1576        # Grid dimensions in z and x directions
dx, dz = 25, 25           # Grid spacing in x and z directions
nt, dt = 3581, 0.001      # Time steps and time interval
nabc = 50                 # Thickness of the absorbing boundary layer
f0 = 15                   # Initial frequency in Hz
free_surface = True       # Enable free surface boundary condition


## Define the True Velocity Model

In [ ]:
# Load the Marmousi model dataset
vp_init = np.load(os.path.join(project_path,"real-data/ref_model1.npz"))["data"].T

# Calculate density (rho) based on velocity
init_rho = np.power(vp_init, 0.25) * 310

# Initialize the AcousticModel with parameters and properties
model = AcousticModel(ox, oz, nx, nz, dx, dz,
                      vp_init, init_rho,
                      vp_grad=False,
                      free_surface=free_surface,
                      abc_type="PML",
                      abc_jerjan_alpha=0.007,
                      nabc=nabc
                    )

# Save the model to a file
model.save(os.path.join(project_path, "model/init_model.npz"))

# Print the model representation
print(model.__repr__())

In [ ]:
# Plot the primary wave velocity (vp) and density (rho) of the model
model._plot_vp_rho(figsize=(12,6),wspace=0.2,cbar_pad_fraction=0.001,cmap='coolwarm',save_path=os.path.join(project_path,"model/init_vp_rho.png"))

## Define the observed System： Survey = Source + Receiver

In [ ]:
src_data = np.load(os.path.join(project_path,"real-data/src_loc.npz"))["data"]
rcv_data = np.load(os.path.join(project_path,"real-data/rcv_loc.npz"))["data"]
rcv_range = np.load(os.path.join(project_path,"real-data/rcv_range.npz"))["data"]
rcv_mask = np.load(os.path.join(project_path,"real-data/rcv_mask.npz"))["data"]
src_data.shape,rcv_data.shape,rcv_range.shape,rcv_mask.shape

In [ ]:
# Define source positions in the model
src_z = src_data[:,1]
src_x = src_data[:,0]

# Generate wavelet for the source
src_t, src_v = wavelet(nt, dt, f0, amp0=1)  # Create time and wavelet amplitude
src_v = integrate.cumtrapz(src_v*-1, axis=-1, initial=0)  # Integrate wavelet to get velocity
source = Source(nt=nt, dt=dt, f0=f0)  # Initialize source object
# Method 2: Loop through each source position to add them individually
for i in range(len(src_x)):
    source.add_source(src_x=src_x[i], src_z=src_z[i], src_wavelet=src_v, src_type="mt", src_mt=np.array([[1,0,0],[0,1,0],[0,0,1]]))


In [ ]:
# Define receiver positions in the model
rcv_z = rcv_data[:,1]+1
rcv_x = rcv_data[:,0]

receiver = Receiver(nt=nt, dt=dt)  # Initialize receiver object

# Method 2: Loop through each receiver position to add them individually
for i in range(len(rcv_x)):
    receiver.add_receiver(rcv_x=rcv_x[i], rcv_z=rcv_z[i], rcv_type="pr")

In [ ]:
# Create a survey object using the defined source and receiver
survey = Survey(source=source, receiver=receiver)

# Print a representation of the survey object to check its configuration
print(survey.__repr__())

In [ ]:
# Plot the wavelet used in the source
source.plot_wavelet(save_path=os.path.join(project_path, "survey/wavelets.png"))

In [ ]:
# Plot the survey configuration over the velocity model
survey.plot(model.vp, cmap='coolwarm', save_path=os.path.join(project_path, "survey/observed_system.png"))

## Define the propagator & Forward Modeling

In [ ]:
# Initialize the wave propagator using the specified model and survey configuration
F = AcousticPropagator(model, survey)

In [ ]:
# Retrieve the damping array from the propagator and plot it to visualize boundary conditions
damp = F.damp
plot_damp(damp, save_path=os.path.join(project_path, "model/boundary_condition.png"))

In [ ]:
# Perform the forward propagation to record waveforms
record_waveform = F.forward()

# Extract recorded pressure wavefield and particle velocities
rcv_p = record_waveform["p"]  # Recorded pressure wavefield
rcv_u = record_waveform["u"]  # Recorded particle velocity in x-direction
rcv_w = record_waveform["w"]  # Recorded particle veloaccity in z-direction

# Extract forward wavefields for analysis
forward_wavefield_p = record_waveform["forward_wavefield_p"]  # Forward pressure wavefield
forward_wavefield_u = record_waveform["forward_wavefield_u"]  # Forward particle velocity wavefield in x
forward_wavefield_w = record_waveform["forward_wavefield_w"]  # Forward particle velocity wavefield in z

In [ ]:
# Create a SeismicData object to store observed data from the survey
d_obs = SeismicData(survey)

# Record the waveform data into the SeismicData object
d_obs.record_data(record_waveform)

# Save the recorded data to a specified file
# d_obs.save(os.path.join(project_path, "waveform/syn_data.npz"))

In [ ]:
obs_p = np.load(os.path.join(project_path,"real-data/obs_data.npz"))["data"]
obs_p_masks = np.load(os.path.join(project_path,"real-data/obs_data_mask.npz"))["data"]
obs_p = np.transpose(obs_p, (0, 2, 1))
obs_p_masks = np.transpose(obs_p_masks, (0, 2, 1))

In [ ]:
rcv_p = rcv_p.cpu()
syn_p = torch.zeros((obs_p.shape[0],obs_p.shape[1],obs_p.shape[2]))

for k in range(rcv_p.shape[0]):
    syn_p[k] = rcv_p[k,...,np.argwhere(rcv_mask[k]).tolist()].squeeze()

syn_p = syn_p.detach().numpy()


In [ ]:
# mute offset
from ADFWI.utils.offset_mute import mute_offset
waveform_mute_offset = 400
receiver_masks_2D = numpy2tensor(rcv_mask) 
syn_p = numpy2tensor(syn_p)
obs_p = numpy2tensor(obs_p)
shot_index = np.arange(src_x.shape[0])
if waveform_mute_offset is not None:
    receiver_mask_2D = receiver_masks_2D[shot_index] # [shot, rcv]
    src_x            = F.src_x.cpu()[shot_index]
    rcv_x_list       = F.rcv_x.cpu()
    rcv_x = torch.zeros(syn_p.shape[0],syn_p.shape[-1])
    for i in range(syn_p.shape[0]):
        rcv_x[i] = rcv_x_list[np.argwhere(receiver_mask_2D[i]).tolist()].squeeze()   
    syn_p = mute_offset(rcv_x,src_x,dx,syn_p,waveform_mute_offset)
    obs_p = mute_offset(rcv_x,src_x,dx,obs_p,waveform_mute_offset)


In [ ]:
from ADFWI.utils.first_arrivel_picking import apply_mute
# mute late window
waveform_mute_late_window = 0.1
if waveform_mute_late_window is not None:
    syn_p_temp = syn_p.clone()
    obs_p_temp = obs_p.clone()
    for i in range(syn_p.shape[0]):
        syn_p[i] = apply_mute(waveform_mute_late_window, syn_p_temp[i], dt)
        obs_p[i] = apply_mute(waveform_mute_late_window, obs_p_temp[i], dt)


In [ ]:
# normalize
def normalize(data):
    mask    = torch.sum(torch.abs(data),axis=1,keepdim=True) == 0
    max_val = torch.max(torch.abs(data),axis=1,keepdim=True).values
    max_val = max_val.masked_fill(mask, 1)
    data = data/max_val
    return data
syn_p = normalize(syn_p)
obs_p = normalize(obs_p)

In [ ]:
shot = 0
fig,axs = plt.subplots(1,1,figsize=(12,8))
# select the trace 
for i in range(0,syn_p.shape[-1]):
    syn_plot = syn_p[shot,:,i]
    obs_plot = obs_p[shot,:,i]
    axs.plot(obs_plot+i,np.arange(obs_p.shape[1])*dt,c='k',linewidth=0.5)
    axs.plot(syn_plot+i,np.arange(syn_p.shape[1])*dt,c='r',linewidth=0.5)
    
# axs.set_xlim(-1,rcv_p.shape[-1]+1)
axs.set_ylim(0, rcv_p.shape[1]*dt)
axs.invert_yaxis()
axs.set_ylabel("Times (s)",fontsize=20)
axs.tick_params(labelsize = 20)
axs.set_xlabel("Receiver ID",fontsize=20)

plt.show()

In [ ]:
fig,axs = plt.subplots(1,1,figsize=(12,5))
shot = 0
trace = 120
obs_plot = obs_p[shot,:,trace]
syn_plot = syn_p[shot,:,trace]
axs.plot(np.arange(obs_p.shape[1])*dt,obs_plot,c='k',linewidth=0.5)
axs.plot(np.arange(syn_p.shape[1])*dt,syn_plot,c='r',linewidth=0.5)
plt.show()

## Visulization the Synthetic Waveform

In [ ]:
normalize = False  # Set normalization for waveform plotting

# Loop over shots to plot the observed waveforms
for i_shot in range(1):  # Currently set to plot only the first shot
    show = (i_shot == 0)  # Show the plot only for the first shot

    # Plot 2D waveform for the specified shot
    d_obs.plot_waveform2D(
        i_shot=i_shot,
        rcv_type="pressure",
        acoustic_or_elastic="acoustic",
        normalize=normalize,
        figsize=(6, 6),
        cmap='coolwarm',
        save_path=os.path.join(project_path, f"waveform/obs_2D_shot_{i_shot}.png"),
        show=show
    )

    # Plot wiggle representation of the waveform for the specified shot
    d_obs.plot_waveform_wiggle(
        i_shot=i_shot,
        rcv_type="pressure",
        acoustic_or_elastic="acoustic",
        normalize=normalize,
        save_path=os.path.join(project_path, f"waveform/obs_wiggle_shot_{i_shot}.png"),
        show=show
    )

In [ ]:
# Plot the waveform trace for a specific shot and trace index
d_obs.plot_waveform_trace(
    i_shot=0,  # Index of the shot to plot (first shot)
    i_trace=10,  # Index of the trace to plot (10th trace)
    normalize=True  # Normalize the waveform for better visualization
)